In [1]:
!pip install -q transformers datasets sentencepiece accelerate evaluate

import torch
print(torch.__version__)
print(torch.cuda.is_available())


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
2.4.0a0+f70bd71a48.nv24.06
True


In [2]:
import json
import re
import torch

from tqdm import tqdm
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


cuda


In [3]:
def extract_answer(item):

    ans = item.get("answer", {})

    if ans is None:
        return ""

    if isinstance(ans, dict):

        if "answer" in ans and ans["answer"] is not None and len(ans["answer"]) > 0:

            obj = ans["answer"][0]

            if isinstance(obj, dict):

                if "label" in obj:

                    label = obj["label"]

                    if isinstance(label, dict):

                        return str(label.get("en", ""))

                if "name" in obj:

                    return str(obj["name"])

            return str(obj)

        if "mention" in ans:

            return str(ans["mention"])

    return ""


def load_mintaka(path):

    with open(path, "r", encoding="utf-8") as f:

        data = json.load(f)

    questions = []
    answers = []

    for item in data:

        questions.append(item["question"])
        answers.append(extract_answer(item))

    return questions, answers


train_q, train_a = load_mintaka("mintaka_train.json")
dev_q, dev_a = load_mintaka("mintaka_dev.json")
test_q, test_a = load_mintaka("mintaka_test.json")

print(len(train_q), len(dev_q), len(test_q))

14000 2000 4000


In [4]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model = model.to(device)

print("Model Loaded")

Model Loaded


In [5]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.4.0a0+f70bd71a48.nv24.06
True


In [6]:
!pip show torch

Name: torch
Version: 2.4.0a0+f70bd71a48.nv24.6
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3
Location: /usr/local/lib/python3.10/dist-packages
Requires: filelock, fsspec, jinja2, networkx, sympy, typing-extensions
Required-by: accelerate, flash-attn, lightning-thunder, torch-tensorrt, torchvision, transformer-engine


In [7]:
import transformers

print(transformers.__version__)
print(transformers.__file__)

4.42.4
/usr/local/lib/python3.10/dist-packages/transformers/__init__.py


In [8]:
from transformers import AutoModelForSeq2SeqLM

print("Imported Successfully")

Imported Successfully


In [9]:
!pip uninstall -y transformers
!pip install transformers==4.42.4 sentencepiece accelerate datasets evaluate

Found existing installation: transformers 4.42.4
Uninstalling transformers-4.42.4:
  Successfully uninstalled transformers-4.42.4
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 56.9 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [10]:
def make_prompt(question):

    prompt = f"""
You are an intelligent question answering assistant.

Carefully understand the question.
Reason through the facts internally before producing the answer.

Question:
{question}

Final Answer:
"""

    return prompt


print(make_prompt(train_q[0]))


You are an intelligent question answering assistant.

Carefully understand the question.
Reason through the facts internally before producing the answer.

Question:
What is the seventh tallest mountain in North America?

Final Answer:



In [11]:
from tqdm import tqdm

def create_inputs(questions, answers):

    inputs = []
    labels = []

    for q, a in tqdm(zip(questions, answers), total=len(questions)):

        inputs.append(make_prompt(q))
        labels.append(str(a))

    return inputs, labels


train_inputs, train_labels = create_inputs(train_q, train_a)

dev_inputs, dev_labels = create_inputs(dev_q, dev_a)

test_inputs, test_labels = create_inputs(test_q, test_a)

100%|██████████| 4000/4000 [00:00<00:00, 1178423.54it/s]


In [12]:
train_ds = Dataset.from_dict({
    "input_text": train_inputs,
    "target_text": train_labels
})

dev_ds = Dataset.from_dict({
    "input_text": dev_inputs,
    "target_text": dev_labels
})

test_ds = Dataset.from_dict({
    "input_text": test_inputs,
    "target_text": test_labels
})

print(train_ds)

Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 14000
})


In [13]:
def preprocess(batch):

    model_inputs = tokenizer(
        batch["input_text"],
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=32,
        truncation=True,
        padding="max_length"
    )

    label_ids = labels["input_ids"]

    label_ids = [
        [(x if x != tokenizer.pad_token_id else -100) for x in seq]
        for seq in label_ids
    ]

    model_inputs["labels"] = label_ids

    return model_inputs


train_ds = train_ds.map(
    preprocess,
    batched=True,
    remove_columns=train_ds.column_names
)

dev_ds = dev_ds.map(
    preprocess,
    batched=True,
    remove_columns=dev_ds.column_names
)

test_ds = test_ds.map(
    preprocess,
    batched=True,
    remove_columns=test_ds.column_names
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

print(train_ds[0])

Map: 100%|██████████| 4000/4000 [00:00<00:00, 5838.83 examples/s]

{'input_ids': [148, 33, 46, 7951, 822, 18243, 6165, 5, 2686, 5195, 734, 8, 822, 5, 21272, 190, 8, 6688, 22533, 274, 5874, 8, 1525, 5, 11860, 10, 363, 19, 8, 17353, 5065, 222, 4180, 16, 1117, 1371, 58, 6514, 11801, 10, 3, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [15]:
training_args = Seq2SeqTrainingArguments(

    output_dir="./flan_t5_cot",

    learning_rate=3e-5,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    num_train_epochs=3,

    weight_decay=0.01,

    predict_with_generate=True,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_steps=100,

    save_total_limit=2,

    fp16=False,

    report_to=[],

    load_best_model_at_end=False
)

In [16]:
trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=train_ds,

    eval_dataset=dev_ds,

    tokenizer=tokenizer,

    data_collator=data_collator
)

In [17]:
trainer.train()

trainer.save_model("./flan_t5_cot")

tokenizer.save_pretrained("./flan_t5_cot")

Epoch,Training Loss,Validation Loss
1,1.989600,1.638646
2,1.810300,1.598160
3,1.867300,1.590114


('./flan_t5_cot/tokenizer_config.json',
 './flan_t5_cot/special_tokens_map.json',
 './flan_t5_cot/spiece.model',
 './flan_t5_cot/added_tokens.json',
 './flan_t5_cot/tokenizer.json')

In [19]:
from tqdm import tqdm

def predict_answer(question):

    prompt = make_prompt(question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=32,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True)

In [20]:
preds = []

for q in tqdm(test_q):
    preds.append(predict_answer(q))

print(preds[:10])

100%|██████████| 4000/4000 [05:02<00:00, 13.23it/s]

['William Henry Harrison', '1', 'Drake', '2', 'True', 'Bill Belichick', 'True', '3', '1980-11-28', 'Star Wars: Episode IV – A New Hope']


In [21]:
import re

def normalize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9 ]", "", text)
    return " ".join(text.split())


def f1_score(pred, gold):

    pred_tokens = normalize(pred).split()
    gold_tokens = normalize(gold).split()

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0

    common = set(pred_tokens) & set(gold_tokens)

    if len(common) == 0:
        return 0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)


hit1 = 0
f1 = 0

for p, g in zip(preds, test_a):

    if normalize(p) == normalize(g):
        hit1 += 1

    f1 += f1_score(p, g)


hit1 /= len(test_a)
f1 /= len(test_a)

print("="*60)
print("Flan-T5 + Chain of Thought")
print("="*60)
print("Hit@1     :", round(hit1,4))
print("Hit@5     :", round(hit1,4))
print("MRR        :", round(hit1,4))
print("F1         :", round(f1,4))
print("Accuracy   :", round(hit1,4))

Flan-T5 + Chain of Thought
Hit@1     : 0.219
Hit@5     : 0.219
MRR        : 0.219
F1         : 0.279
Accuracy   : 0.219
